# HP-tuning stability check

Does it matter where the 10-day tuning block sits?  
For a subset of stocks the tune → freeze → walk-forward pipeline is run with the block at 0%, 25%, 50% and 75% of the sample, evaluating on the days after the block.

In [1]:
import os, sys, json, time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.path.dirname(os.getcwd()))
import importlib
import utils.data_processing as du
import utils.execution as execution
import utils.pipeline as pipeline
import utils.workers as workers

for m in (du, pipeline, execution, workers):
    importlib.reload(m)


In [ ]:
# Config (same tuning protocol as xgboost.ipynb, plus OFFSETS)

STABILITY_SYMBOLS = [
 'Adidas',
 'Qiagen']

OFFSETS = [0.00, 0.33, 0.67] # tuning-block start

TUNE_BLOCK_LEN = 3
TRAIN_DAYS = 1

N_TRIALS = 1
N_PAIRS = 2

SEED = 0

N_PROC = 1
N_JOBS = -1

HORIZONS = ["100ms", "2s", "30s", "5m"]

# name -> (distribution, low, high)
SEARCH_SPACE = {
    "max_depth": ("int", 1, 6),
    "learning_rate": ("log", 0.01, 0.3),
    "min_child_weight": ("logint", 5, 100),
    "subsample": ("uniform", 0.5, 1.0),
    "colsample_bytree": ("uniform", 0.5, 1.0),
    "reg_lambda": ("log", 0.01, 10.0),
}

EARLY_STOPPING = dict(n_estimators=2000, early_stopping_rounds=50, eval_metric="rmse")

XGB_PARAMS = dict(
    tree_method="hist",
    max_bin=128,
    n_jobs=N_JOBS,
    random_state=0,
)

DEVICE = execution.select_device()
print(f"XGBoost device: {DEVICE}")

XGBoost device: cpu


In [ ]:
# Tuning block and eval window per offset
PARENT = os.path.dirname(os.getcwd())
DATA_ROOT = f"{PARENT}/data/processed"
MODEL_ROOT = f"{PARENT}/model_outputs/XGBoost"
STABILITY_DIR = f"{MODEL_ROOT}/runs/stability_seed{SEED}"

sample_cols = pd.read_parquet(f"{DATA_ROOT}/{STABILITY_SYMBOLS[0]}/{du.SAMPLE_DATES[0]}.parquet").columns
FEATURE_COLS = list(sample_cols[sample_cols.str.startswith("F_")])
TARGET_COLS = [c for c in sample_cols if c.startswith("T_") and c.rsplit("_", 1)[-1] in HORIZONS]

ALL_DATES = list(du.SAMPLE_DATES)
BLOCKS = {}
print(f"totals: trials {len(TARGET_COLS)} targets/stock (every offset)")
for off in OFFSETS:
    start = round(off * len(ALL_DATES))
    tune_dates = ALL_DATES[start:start + TUNE_BLOCK_LEN]
    eval_dates = ALL_DATES[start + TUNE_BLOCK_LEN:]
    if len(tune_dates) < TUNE_BLOCK_LEN or len(eval_dates) < 2:
        raise ValueError(f"offset {off:.0%}: not enough days (tune {len(tune_dates)}, eval {len(eval_dates)})")
    BLOCKS[off] = {"tune_dates": tune_dates, "eval_dates": eval_dates}
    print(f"{off:>4.0%}: tune {tune_dates[0]} .. {tune_dates[-1]}, "
          f"eval {eval_dates[0]} .. {eval_dates[-1]} ({len(eval_dates)} days), "
          f"partial {len(eval_dates) - TRAIN_DAYS} days/stock")


In [ ]:
# Each offset is one run dir with its own manifest.
GLOBAL_START = time.perf_counter()


def update_manifest(run_dir, **fields):
    with open(f"{run_dir}/manifest.json") as f:
        manifest = json.load(f)
    manifest.update(fields)
    with open(f"{run_dir}/manifest.json", "w") as f:
        json.dump(manifest, f, indent=2, default=str)
    return manifest


for off in OFFSETS:
    off_pct = int(off * 100)
    name = f"stability_off{off_pct}"
    run_dir = f"{STABILITY_DIR}/{name}"
    tune_dates = BLOCKS[off]["tune_dates"]

    if not os.path.exists(f"{run_dir}/manifest.json"):
        # nested name so start_run puts the run dir under STABILITY_DIR
        pipeline.start_run(MODEL_ROOT, f"stability_seed{SEED}/{name}", {
            "status": "tuning",
            "purpose": f"HP-tuning stability check, tuning block at {off:.0%} of the sample",
            "offset": off,
            "symbols": STABILITY_SYMBOLS,
            "horizons": HORIZONS,
            "feature_cols": FEATURE_COLS,
            "target_cols": TARGET_COLS,
            "train_days": TRAIN_DAYS,
            "tune_dates": list(tune_dates),
            "xgb_params": XGB_PARAMS,
            "target_scale": pipeline.TARGET_SCALE,
            "tuning": {"seed": SEED, "n_trials": N_TRIALS, "n_pairs": N_PAIRS,
                       "search_space": SEARCH_SPACE, "early_stopping": EARLY_STOPPING},
            "params": None,
        })

    # tuning (skips stocks with a trials checkpoint)
    todo = [s for s in STABILITY_SYMBOLS if not os.path.exists(f"{run_dir}/trials/{s}.parquet")]
    codes = execution.run_parallel(workers.tune_xgb, {s: (run_dir, s, DEVICE) for s in todo}, n_proc=N_PROC)

    # freeze winners into the manifest
    all_trials = pd.concat([pd.read_parquet(f"{run_dir}/trials/{s}.parquet") for s in STABILITY_SYMBOLS], ignore_index=True)
    best_params = workers.freeze_winners(all_trials, SEARCH_SPACE)
    update_manifest(run_dir, params=best_params, status="running")

    # walk-forward on the days after the block (skips stocks with a partial checkpoint)
    todo = [s for s in STABILITY_SYMBOLS if not os.path.exists(f"{run_dir}/partial/{s}.parquet")]
    codes = execution.run_parallel(workers.run_xgb, {s: (run_dir, s, DEVICE) for s in todo}, n_proc=N_PROC)

    daily = pd.concat([pd.read_parquet(f"{run_dir}/partial/{s}.parquet") for s in STABILITY_SYMBOLS], ignore_index=True)
    daily.to_parquet(f"{run_dir}/daily_diagnostics.parquet", index=False)
    manifest = update_manifest(run_dir, status="complete")
    update_manifest(run_dir, runtime_seconds=round(
        (datetime.now(timezone.utc) - datetime.fromisoformat(manifest["created_at"])).total_seconds(), 1))
    print(f"{name}: done, {time.perf_counter()-GLOBAL_START:.0f}s elapsed")

# combined file for the analysis cells
daily_results = pd.concat(
    [pd.read_parquet(f"{STABILITY_DIR}/stability_off{int(off * 100)}/daily_diagnostics.parquet")
       .assign(model_leg="xgb", offset_pct=int(off * 100))
     for off in OFFSETS],
    ignore_index=True,
)
daily_results.to_parquet(f"{STABILITY_DIR}/daily_diagnostics.parquet", index=False)
print(f"TOTAL STABILITY-CHECK TIME: {time.perf_counter()-GLOBAL_START:.2f}s")

In [ ]:
# Mean MSE ratio per measure x horizon x offset, full and shared eval window
daily_out = pd.read_parquet(f"{STABILITY_DIR}/daily_diagnostics.parquet")
daily_out["horizon"] = daily_out["target"].str.rsplit("_", n=1).str[-1]
daily_out["horizon"] = pd.Categorical(daily_out["horizon"], categories=HORIZONS, ordered=True)
daily_out["measure"] = (daily_out["target"]
                        .str.removeprefix("T_")
                        .str.rsplit("_", n=1).str[0])

MEASURES = sorted(daily_out["measure"].unique())
MEASURE_COLORS = dict(zip(MEASURES, plt.rcParams["axes.prop_cycle"].by_key()["color"]))

display(
    daily_out.pivot_table(index=["measure", "horizon"], columns="offset_pct",
                          values="mse_ratio", observed=True).round(4)
)

shared = daily_out[daily_out["test_day"].isin(BLOCKS[OFFSETS[-1]]["eval_dates"])]
display(
    shared.pivot_table(index=["measure", "horizon"], columns="offset_pct",
                       values="mse_ratio", observed=True).round(4)
)

## XGB vs OLS

Paired daily Diebold-Mariano test (Newey-West) on the daily cross-stock mean of
`d = mse_ratio_xgb − mse_ratio_ols`. XGB leg is a concat of the offset runs:
each test day uses the most recently tuned hyperparameters.

In the figures, a (measure, horizon) cell is a *dead zone* when neither model beats
the zero-return benchmark on average (mean MSE ratio ≤ 1 for both); those cells are
hatched in the heatmap. Stars in the heatmap come from the DM test
(\* p<0.05, \*\* p<0.01, \*\*\* p<0.001), shown only where OLS is better on average.

In [ ]:
DM_REGRESSION_RUN_ID = "05"   # run under model_outputs/Regression/runs/

ols_daily = pd.read_parquet(f"{PARENT}/model_outputs/Regression/runs/{DM_REGRESSION_RUN_ID}/daily_diagnostics.parquet")

# composite: highest offset whose eval window contains the day
xgb_leg = daily_out.loc[daily_out.groupby(["symbol", "target", "test_day"])["offset_pct"].idxmax()]
print(xgb_leg.groupby("offset_pct")["test_day"].nunique())


def newey_west_tstat(d: np.ndarray) -> tuple:
    """Mean, NW se, t-stat, p-value (Bartlett kernel, lag = floor(1.5 T^(1/3)))."""
    from scipy import stats
    T = len(d)
    mean = d.mean()
    u = d - mean
    lag = int(np.floor(1.5 * T ** (1 / 3)))
    s = float(u @ u) / T
    for k in range(1, lag + 1):
        s += 2 * (1 - k / (lag + 1)) * float(u[k:] @ u[:-k]) / T
    se = np.sqrt(s / T)
    t = mean / se
    return mean, se, t, 2 * stats.norm.sf(abs(t))


def dm_table(paired):
    """DM test per measure x horizon on the daily cross-stock mean of paired['d']."""
    rows = []
    for (measure, horizon), g in paired.groupby(["measure", "horizon"], observed=True):
        d = g.groupby("test_day")["d"].mean().sort_index().to_numpy()
        mean, se, t, p = newey_west_tstat(d)
        rows.append({"measure": measure, "horizon": horizon, "n_days": len(d),
                     "mean_diff": mean, "se": se, "t": t, "p": p})
    out = pd.DataFrame(rows)
    out["horizon"] = pd.Categorical(out["horizon"], categories=HORIZONS, ordered=True)
    return out.sort_values(["measure", "horizon"], ignore_index=True)


def dm_errorbar_plot(df, ylabel, title):
    fig, ax = plt.subplots(figsize=(6.5, 4))
    x = np.arange(len(HORIZONS))
    width = 0.8 / len(MEASURES)
    for k, measure in enumerate(MEASURES):
        g = df[df["measure"] == measure].set_index("horizon").reindex(HORIZONS)
        ax.errorbar(x + (k - (len(MEASURES) - 1) / 2) * width, g["mean_diff"],
                    yerr=1.96 * g["se"], fmt="o", ms=4, lw=1.2,
                    color=MEASURE_COLORS[measure], label=measure)
    ax.axhline(0.0, color="gray", lw=1, ls="--")
    ax.set_xticks(x, HORIZONS)
    ax.set_xlabel("horizon")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(title="price measure", frameon=False, fontsize=8, title_fontsize=8)
    fig.tight_layout()


paired = xgb_leg.merge(
    ols_daily[["symbol", "test_day", "target", "mse_ratio"]],
    on=["symbol", "test_day", "target"],
    suffixes=("_xgb", "_ols"),
    validate="one_to_one",
)
paired["d"] = paired["mse_ratio_xgb"] - paired["mse_ratio_ols"]

cell_means = (paired.groupby(["measure", "horizon"], observed=True)
              [["mse_ratio_xgb", "mse_ratio_ols"]].mean())
DEAD_ZONES = set(cell_means.query("mse_ratio_xgb <= 1 and mse_ratio_ols <= 1").index)
print("\nDead zones (neither OLS nor the composite XGB beats the benchmark):")
for m, h in sorted(DEAD_ZONES):
    print(f"  {m} @ {h}")

dm = dm_table(paired)
dm = dm.merge(
    paired.groupby(["measure", "horizon"], observed=True)
          .agg(ols_win_rate=("d", lambda s: (s <= 0).mean()),
               ratio_ols=("mse_ratio_ols", "mean"),
               ratio_xgb=("mse_ratio_xgb", "mean"))
          .reset_index(),
    on=["measure", "horizon"],
)
dm["dead_zone"] = [mh in DEAD_ZONES for mh in zip(dm["measure"], dm["horizon"])]
display(dm.round(4))

dm_errorbar_plot(dm, "mean daily MSE-ratio difference\n(XGB − OLS, 95% NW CI)",
                 "XGB (freshest HPs) vs OLS")

In [ ]:
# Heatmap: share of stock-days where OLS >= XGB (encodings: see markdown above)
heat = (dm.pivot(index="measure", columns="horizon", values="ols_win_rate")
          .reindex(columns=HORIZONS).sort_index())
pvals = dm.pivot(index="measure", columns="horizon", values="p").reindex(
    index=heat.index, columns=HORIZONS)
means = dm.pivot(index="measure", columns="horizon", values="mean_diff").reindex(
    index=heat.index, columns=HORIZONS)
dead = dm.pivot(index="measure", columns="horizon", values="dead_zone").reindex(
    index=heat.index, columns=HORIZONS)


def stars(p, mean_diff):
    if mean_diff >= 0:
        return ""
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""


fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(heat.to_numpy(), cmap="RdBu", vmin=0.0, vmax=1.0, aspect="auto")
for r in range(heat.shape[0]):
    for c in range(heat.shape[1]):
        v = heat.iat[r, c]
        if dead.iat[r, c]:
            ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, fill=False,
                                       hatch="///", edgecolor="gray", lw=0))
        ax.text(c, r, f"{v:.0%}{stars(pvals.iat[r, c], means.iat[r, c])}",
                ha="center", va="center", fontsize=9,
                color="white" if abs(v - 0.5) > 0.3 else "black")
ax.set_xticks(range(len(HORIZONS)), HORIZONS)
ax.set_yticks(range(len(heat)), heat.index)
ax.set_xlabel("horizon")
ax.set_title("OLS daily win rate vs XGB")
fig.colorbar(im, ax=ax, label="OLS win rate", fraction=0.04, pad=0.02)
fig.tight_layout()

In [ ]:
# Does re-tuning help? DM test of the composite vs the 0%-tuned run on the
# days where they differ: d = mse_ratio_fresh - mse_ratio_stale
fresh_days = set(xgb_leg.loc[xgb_leg["offset_pct"] > 0, "test_day"])
stale = daily_out[(daily_out["offset_pct"] == 0)
                  & daily_out["test_day"].isin(fresh_days)]

paired_x = xgb_leg[xgb_leg["test_day"].isin(fresh_days)].merge(
    stale[["symbol", "test_day", "target", "mse_ratio"]],
    on=["symbol", "test_day", "target"],
    suffixes=("_fresh", "_stale"),
    validate="one_to_one",
)
paired_x["d"] = paired_x["mse_ratio_fresh"] - paired_x["mse_ratio_stale"]

retune = dm_table(paired_x)
retune = retune.merge(
    paired_x.groupby(["measure", "horizon"], observed=True)["d"]
            .agg(fresh_win_rate=lambda s: (s > 0).mean()).reset_index(),
    on=["measure", "horizon"],
)
display(retune.round(4))

dm_errorbar_plot(retune, "mean daily MSE-ratio difference\n(fresh − stale HPs, 95% NW CI)",
                 "Value of re-tuning: freshest HPs vs tuned once at 0%")

In [ ]:
# Winning hyperparameters by horizon, and HP-landscape flatness
# (winner vs median trial score: small gap = tuning barely matters)
trials_all = pd.concat(
    [pd.read_parquet(f"{STABILITY_DIR}/stability_off{int(off * 100)}/trials/{symbol}.parquet")
     .assign(offset_pct=int(off * 100))
     for off in OFFSETS for symbol in STABILITY_SYMBOLS],
    ignore_index=True,
)
trials_all["horizon"] = pd.Categorical(
    trials_all["target"].str.rsplit("_", n=1).str[-1], categories=HORIZONS, ordered=True)

winners = trials_all.loc[
    trials_all.groupby(["offset_pct", "symbol", "target"])["mean_mse_ratio"].idxmax()
].copy()

rng = np.random.default_rng(0)   # jitter only


def jitter_plot(ax, df, col):
    """Per-horizon jittered scatter of df[col] with a median bar."""
    for h, horizon in enumerate(HORIZONS):
        vals = df.loc[df["horizon"] == horizon, col].to_numpy(dtype=float)
        ax.scatter(h + rng.uniform(-0.18, 0.18, len(vals)), vals,
                   s=12, alpha=0.5, color="C0", edgecolors="none")
        ax.hlines(np.median(vals), h - 0.25, h + 0.25, color="C1", lw=2)
    ax.set_xticks(range(len(HORIZONS)), HORIZONS)


PLOT_PARAMS = ["max_depth", "learning_rate", "min_child_weight",
               "reg_lambda", "n_estimators_frozen", "subsample"]
LOG_SCALE = {"learning_rate", "min_child_weight", "reg_lambda", "n_estimators_frozen"}

fig, axes = plt.subplots(2, 3, figsize=(11, 6))
for ax, param in zip(axes.flat, PLOT_PARAMS):
    jitter_plot(ax, winners, param)
    if param in LOG_SCALE:
        ax.set_yscale("log")
    if param in SEARCH_SPACE:
        for bound in SEARCH_SPACE[param][1:]:
            ax.axhline(bound, color="gray", lw=0.8, ls=":")
    ax.set_title(param, fontsize=10)
fig.suptitle("Winning configs per horizon (orange = median, dotted = search bounds)", y=1.0)
fig.tight_layout()

flat = (trials_all.groupby(["offset_pct", "symbol", "target", "horizon"], observed=True)
        ["mean_mse_ratio"]
        .agg(best="max", median="median", q25=lambda s: s.quantile(0.25))
        .reset_index())
flat["best_minus_median"] = flat["best"] - flat["median"]
flat["best_minus_q25"] = flat["best"] - flat["q25"]

display(flat.groupby("horizon", observed=True)
        [["best", "median", "best_minus_median", "best_minus_q25"]]
        .mean().round(4))

fig, ax = plt.subplots(figsize=(6.5, 4))
jitter_plot(ax, flat, "best_minus_median")
ax.axhline(0.0, color="gray", lw=1, ls="--")
ax.set_xlabel("horizon")
ax.set_ylabel("winner − median trial\n(validation mean MSE ratio)")
ax.set_title("HP-landscape flatness")
fig.tight_layout()